# Tour --- Akgul et al. 2026: Sparse Policy Selection for LLM Reasoning> Companion notebook to `tour.md` and `tour.tex` in this directory.> arXiv: [2605.06241](https://arxiv.org/abs/2605.06241)This notebook lets you (a) quickly see the navigation map, and (b) run the MEASUREMENT-mode improvements from `improvements/`.

## Section 1 --- Reader's Contract**Audience:** mathematicians, ML researchers in adjacent subfields, engineers comfortable with per-token training equations.**Background assumed:** linear algebra, multivariate calculus through the Jacobian, probability through expectation/variance, Shannon information through entropy & KL.**Recommended foundation tour:** [math-grad](https://github.com/pleyva2004/math-foundations/blob/main/tours/math-grad.md). Other tours: [novice](https://github.com/pleyva2004/math-foundations/blob/main/tours/novice.md), [cs-undergrad](https://github.com/pleyva2004/math-foundations/blob/main/tours/cs-undergrad.md), [researcher](https://github.com/pleyva2004/math-foundations/blob/main/tours/researcher.md).**Time budget:** ~3-4 hours for the first focused pass.**Elevator:** RL on verifiable rewards for math reasoning is sparse, entropy-gated re-selection at the top 2-8% of token positions, where the RL choice sits at base-rank ~2.2 and the whole effect is reproducible offline by ReasonMaxxer.

## Section 2 --- Foundations WalkTopological order from the math-foundations manifest. Two nodes earn "drill into proof": Shannon entropy (the paper's organising scalar) and KL divergence (the asymmetry choice in the anchoring loss is load-bearing).

In [ ]:
# Tour navigation summary --- prints which foundation concepts and paper concepts
# this tour walks through, in order.

import json, os, urllib.parse

NOTATION_PATH = "../notation.json"  # may not exist yet; degrade gracefully

foundations = [
    ("06", "linear-maps",            "Linear Maps and Matrices",  "intermediate", "skim"),
    ("07", "eigenvalues",            "Eigenvalues",               "intermediate", "skim"),
    ("14", "gradient-jacobian",      "Gradient and Jacobian",     "advanced",     "read"),
    ("22", "random-variables",       "Random Variables",          "advanced",     "skim"),
    ("23", "distributions",          "Probability Distributions", "advanced",     "read"),
    ("25", "expectation",            "Expectation",               "advanced",     "read"),
    ("26", "variance-covariance",    "Variance and Covariance",   "advanced",     "skim"),
    ("36", "shannon-entropy",        "Shannon Entropy",           "advanced",     "DRILL"),
    ("37", "cross-entropy",          "Cross-Entropy",             "advanced",     "read"),
    ("38", "kl-divergence",          "KL Divergence",             "advanced",     "DRILL"),
]

paper_concepts = [
    "P1 token-level entropy H_t",
    "P2 decision-point set D = {t : H_t > tau}",
    "P3 GRPO advantage normalisation",
    "P4 reranked vs shifted classification",
    "P5 contrastive loss on decision points",
    "P6 base-anchoring KL on non-decision points",
    "P7 KL-clipped policy ratio (mostly discarded)",
    "P8 rank-32 LoRA on QKVO",
    "P9 weight tying with input embedding",
]

print("FOUNDATIONS WALK")
print("-" * 60)
for nid, slug, title, level, pacing in foundations:
    print(f"  [{nid}] {title:30s} {level:13s} {pacing}")
print()
print("PAPER CONCEPTS WALK")
print("-" * 60)
for c in paper_concepts:
    print(f"  {c}")

# Try to load notation.json if it exists
if os.path.exists(NOTATION_PATH):
    with open(NOTATION_PATH) as f:
        notation = json.load(f)
    print()
    print("NOTATION SUMMARY (from notation.json)")
    print("-" * 60)
    for k, v in list(notation.items())[:8]:
        print(f"  {k}: {v}")
else:
    print()
    print("(notation.json not present yet --- skipping notation summary)")


## Section 3 --- Paper Concepts WalkSee `tour.md` Section 3 for the full table. The objects above each carry a pointer into `02-math-deep-dive.md` for the equation/section.

## Section 4 --- Improvements WalkValidation modes:- **PROOF** --- LaTeX in `proofs/`- **MEASUREMENT** --- Python in `improvements/` with `measure()` function| ID | Class | Title | Mode | Artifact ||---|---|---|---|---|| M1 | Math | Lipschitz bound on rank displacement | PROOF | `proofs/lipschitz-rank-displacement.tex` || M2 | Math | Info-theoretic top-k tightening | PROOF (deferred) | `proofs/info-theoretic-top-k.tex` || C1 | Code | Quantile-based gating tau | MEASUREMENT | `improvements/adaptive-tau.py` || C2 | Code | Single-pass entropy scoring | MEASUREMENT | `improvements/streaming-entropy.py` || E1 | Exp | OOD stress probe | MEASUREMENT (deferred) | -- || E2 | Exp | Non-verifiable task probe | MEASUREMENT (deferred) | -- || T1 | Theory | Active learning equivalence | PROOF (deferred) | `proofs/active-learning-equivalence.tex` || T2 | Theory | LoRA delta as activation steering | PROOF | `proofs/lora-equiv-steering.tex` |

### MEASUREMENT runsThe two cells below invoke `measure()` from each MEASUREMENT-mode improvement. Sibling subagents in this batch produce the implementations; if those files do not exist yet, the cells will print a `ModuleNotFoundError` and you can re-run after the implementations land.

In [ ]:
# C1 --- adaptive-tau (quantile-based gating)
import sys
sys.path.append("../improvements")
try:
    from adaptive_tau import measure
    result = measure()
    print("adaptive_tau.measure() ->")
    print(result)
except Exception as e:
    print(f"NOT YET RUNNABLE: {type(e).__name__}: {e}")
    print("Expected output shape: {'fixed_tau_gated_pct': [...], 'quantile_gated_pct': [...], 'stability_ratio': ~7x}")


In [ ]:
# C2 --- streaming-entropy (single-pass scoring)
import sys
sys.path.append("../improvements")
try:
    from streaming_entropy import measure
    result = measure()
    print("streaming_entropy.measure() ->")
    print(result)
except Exception as e:
    print(f"NOT YET RUNNABLE: {type(e).__name__}: {e}")
    print("Expected output shape: {'naive_ms': ..., 'streaming_ms': ..., 'speedup': ~2.0, 'max_abs_diff': <1e-6}")


## Section 5 --- What to Do Next**(a) Most personally promising proposal:** M1 (Lipschitz bound on rank displacement). Cleanest mathematician-flavoured contribution; explains an existing empirical regularity (rank ~2.2 across four pairs) via a short perturbation argument on the softmax Jacobian.**(b) Single experiment to settle the most ambiguous claim:** necessity of entropy-gated correction. Replicate ReasonMaxxer with $D^{(i)}$ replaced by (i) a random same-size subset, (ii) the complement $\{t : H_t < \tau\}$. If pass@1 collapses on both, necessity is established. ~1 weekend of compute on a 1.5B model.**(c) Follow-on paper direction:** bridge to mechanistic interpretability via T2. If the LoRA delta is a steering vector, extract it directly from $\pi_{\text{RL}} - \pi_{\text{base}}$, test linear composition across skills, and eliminate RL entirely. Working title: *"RL for Reasoning is Activation Steering in Disguise --- and Here's the Vector."*